# 09 - Comites multi-vainqueurs : STV, Monroe et Chamberlin-Courant (twin C# .NET)

**Navigation** : [Predecessor: [ChoixSocial-08-Kemeny-RankedPairs-Dodgson-Python.ipynb](ChoixSocial-08-Kemeny-RankedPairs-Dodgson-Python.ipynb)] | [ChoixSocial-10-Manipulation-Strategique-MultiWinner-Python.ipynb >>](ChoixSocial-10-Manipulation-Strategique-MultiWinner-Python.ipynb) | [Index](../README.md)

> Jumeau C# .NET de `GameTheory/SocialChoice/ChoixSocial-09-Committees-STV-Monroe-ChamberlinCourant-Python.ipynb` (Python, `z3`). Meme fil pedagogique, bibliotheques .NET natives (`Microsoft.Z3` pour le solveur, BCL/LINQ pour les regles).

**Serie SocialChoice -- notebook 09.** Trois regles representatives du **chapitre 9 (Lang et Xia, p. 207-210)** du *Handbook of Computational Social Choice* : **STV** (Hare 1857, polynomiale), **Monroe** (1995, NP-dur), **Chamberlin-Courant** (1983, NP-dur). (Les regles multi-winners sont aussi traitees en article par Elkind, Faliszewski, Skowron et Slinko, *Properties of Multiwinner Voting Rules*, Social Choice and Welfare 48(3):599-632, 2017 -- ce n'est **pas** un chapitre du Handbook, le Handbook le cite sous "Elkind et al. 2014a".)

Carnet jumeau de SC-08 (Kemeny / Ranked Pairs / Dodgson) : meme pattern, autre chapitre du Handbook, Z3 confronte l'enumeration brute sur les deux regles NP-dures.

Ce que ce notebook fait, a echelle de jouet (n <= 6 votants, m <= 5 candidats, k <= 3) :

1. definir un **profil ordonne** (bulletins = permutations completes) et la taille k du comite ;
2. implementer **STV (Hare)** au quota de Droop : algorithme polynomial, election des qu'un candidat atteint le quota, elimination du moins bien classe seulement en l'absence de candidat au quota ;
3. implementer **Monroe (1995)** par backtracking : assignation proportionnelle multi-winners ou chaque gagnant represente approximativement le meme nombre d'electeurs (`floor(n/k)` ou `ceil(n/k)`) ;
4. implementer **Chamberlin-Courant (1983)** sous score Borda : assignation representationnelle ou chaque gagnant represente un sous-ensemble d'electeurs, satisfaction visee maximale ;
5. confronter **Z3** (`Microsoft.Z3` : `Optimize` + `Int` + contraintes cardinalite) a l'enumeration brute-force pour Monroe et Chamberlin-Courant sur petits profils ;
6. comparer les trois regles sur un fil rouge structure + une experimentation aleatoire.

Prerequis : 01-Arrow, ChoixSocial-03-Voting-Methods, 04-SAT-Z3, ChoixSocial-07-Committees-Core, ChoixSocial-08-Kemeny-RankedPairs-Dodgson.


## 1. Cadre : profil ordonne et taille de comite

Une instance est $I = (R, k)$ ou $R = (R_1, \dots, R_n)$ est un profil de preferences : chaque votant $i$ ordonne completement les $m$ candidats, $R_i \in S_m$. On elit un comite $W$ de taille $k$, $W \subseteq C$, $|W| = k$.

**Utilite de representation**. Chaque votant est represente par le candidat le mieux place dans $W$ pour ses preferences. Pour Chamberlin-Courant, on utilise souvent le score de **Borda** : si le candidat en position $t$ (1-indexe, le premier = meilleur) est dans $W$, le votant gagne $m - t$ points. Pour STV et Monroe, le rang absolu du candidat gagnant dans le classement du votant suffit.

**Exemple fil rouge** : $n = 5$ votants, $m = 4$ candidats $\{a, b, c, d\}$, $k = 2$ elus. Bulletins :

- $v_1$ : $a > b > c > d$
- $v_2$ : $a > b > c > d$
- $v_3$ : $b > c > d > a$
- $v_4$ : $c > d > a > b$
- $v_5$ : $d > a > b > c$

Les preferences sont correlees (les cinq votants privilegient globalement $a$ puis $b$) -- STV elit $a$ et $c$, Monroe doit aussi equilibrer les representes, Chamberlin-Courant cherche la paire qui maximise la satisfaction Borda agregee.


In [1]:
// Fil rouge : 5 votants, 4 candidats, k = 2 elus -- helpers rang/score_borda et affichage
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;

public static class Sc09
{
    // Profil ordonne du fil rouge : chaque bulletin est une permutation complete.
    public static readonly List<string[]> R = new List<string[]>
    {
        new[] { "a", "b", "c", "d" },  // v1 : a > b > c > d
        new[] { "a", "b", "c", "d" },  // v2
        new[] { "b", "c", "d", "a" },  // v3
        new[] { "c", "d", "a", "b" },  // v4
        new[] { "d", "a", "b", "c" },  // v5
    };

    public static int N => R.Count;      // nombre de votants
    public static int M => R[0].Length;  // nombre de candidats
    public static int K => 2;            // taille du comite
    public static List<string> Cands => R[0].ToList();

    // Position 1-indexee de c dans le bulletin du votant (1 = meilleur).
    public static int Rang(string[] votant, string c) => Array.IndexOf(votant, c) + 1;

    // Score Borda du votant pour le comite W : meilleur candidat dans W.
    // m = nombre de candidats du profil (PAS une variable globale) :
    // le test reste possible sur des profils aleatoires de taille differente.
    public static int ScoreBorda(string[] votant, IEnumerable<string> W, int m)
    {
        var positions = W.Where(c => votant.Contains(c))
                         .Select(c => Array.IndexOf(votant, c) + 1)
                         .ToList();
        if (positions.Count == 0) return 0;
        int best = positions.Min();
        return m - best;   // candidat en t -> m - t points Borda
    }

    // Affichage "a la Python" des listes de candidats.
    public static string Fmt(IEnumerable<string> xs)
        => "[" + string.Join(", ", xs.Select(x => $"'{x}'")) + "]";

    // Candidats distincts du profil, tries (ordre alphabetique).
    public static List<string> AllCands(IEnumerable<string[]> profile)
        => profile.SelectMany(r => r).Distinct().OrderBy(c => c, StringComparer.Ordinal).ToList();

    // Combinaisons de taille k, dans l'ordre lexicographique (comme itertools.combinations).
    public static List<List<string>> Combinations(List<string> items, int k)
    {
        var result = new List<List<string>>();
        void Rec(int start, List<string> cur)
        {
            if (cur.Count == k) { result.Add(cur.ToList()); return; }
            for (int i = start; i < items.Count; i++)
            {
                cur.Add(items[i]);
                Rec(i + 1, cur);
                cur.RemoveAt(cur.Count - 1);
            }
        }
        Rec(0, new List<string>());
        return result;
    }
}

var R = Sc09.R;
int N = Sc09.N;
int m = Sc09.M;
int K = Sc09.K;
var CANDS = Sc09.Cands;

Console.WriteLine($"Profil : {N} votants, {m} candidats, k = {K}");
Console.WriteLine($"Candidats : {Sc09.Fmt(CANDS)}");
for (int i = 0; i < R.Count; i++)
{
    int scoreMax = Sc09.ScoreBorda(R[i], CANDS, m);
    Console.WriteLine($"  v{i + 1} : {string.Join(" > ", R[i])}  (Borda max = {scoreMax})");
}

// Premier constat : combien de votants obtiennent quel rang pour chaque candidat ?
Console.WriteLine("Premier constat : rangs obtenus par chaque candidat");
foreach (var c in CANDS)
{
    var rs = R.Select(r => Sc09.Rang(r, c)).OrderBy(x => x).ToList();
    Console.WriteLine($"  Rangs de {c} : [{string.Join(", ", rs)}]");
}


Profil : 5 votants, 4 candidats, k = 2


Candidats : ['a', 'b', 'c', 'd']


  v1 : a > b > c > d  (Borda max = 3)


  v2 : a > b > c > d  (Borda max = 3)


  v3 : b > c > d > a  (Borda max = 3)


  v4 : c > d > a > b  (Borda max = 3)


... (2 outputs left out)

  Rangs de a : [1, 1, 2, 3, 4]


  Rangs de b : [1, 2, 2, 3, 4]


  Rangs de c : [1, 2, 3, 3, 4]


  Rangs de d : [1, 2, 3, 4, 4]


## 2. STV (Single Transferable Vote, Hare 1857)

STV est une regle multi-winners **polynomiale** qui transpose le deuxieme tour francais a un cadre proportionnel. Le **quota de Droop** definit le seuil d'election :

$$q = \left\lfloor \frac{n}{k+1} \right\rfloor + 1.$$

Au quota de Droop, un candidat elu dispose d'assez de surplus pour transferer un excedent a un autre candidat sans permettre a l'election de deux candidats superieurs au quota par les memes voix.

**Algorithme (election d'abord, elimination en secours)** :

1. Chaque votant dispose d'un suffrage attribue a son candidat prefere encore en lice (ni elu ni elimine).
2. **Election** : des qu'un candidat atteint ou depasse le quota $q$, il est elu. Son surplus est transfere : chaque votant qui le soutenait transfere une fraction $1 - q/v$ de sa voix (surplus fractionnaire, methode de Gregory 1880 ; Hare 1857 designe le scrutin lui-meme) a son prochain prefere encore en lice.
3. **Elimination** : si aucun candidat n'atteint le quota, le candidat avec le moins de voix est elimine. Le transfert est alors **integral** : seuls les votants dont le porteur courant est l'elimine transferent leur voix au prochain prefere encore en lice -- le decompte total reste constant (une voix ne se compte jamais deux fois, et une voix ne disparait jamais parce que son porteur a ete elimine).
4. On repete jusqu'a elire $k$ candidats, en eliminant un candidat **seulement tant qu'il reste assez de survivants** pour occuper les sieges encore vacants.

L'ordre **election avant elimination** n'est pas cosmetique : eliminer d'abord un candidat sous-quota alors qu'un autre candidat a deja atteint le quota peut ecarter du comite un candidat qui aurait ete elu.

**Convention de fin de scrutin.** A force d'eliminer, on peut epuiser les candidats avant d'avoir pourvu tous les sieges : il faut alors decider qui les occupe. On **complete le comite par les survivants** (ceux qui n'ont pas ete elimines), classes par votes decroissants puis par ordre alphabetique. Un candidat **elimine** ne peut pas etre reintegre : tous les elimines sont a score nul, les classer entre eux reviendrait a les departager **alphabetiquement** au lieu de les classer par popularite -- le comite dependrait alors de l'ordre des lettres, et un vote parasite pourrait faire entrer un candidat par ordre alphabetique au lieu de son merite. Avec cette convention, le comite rendu a toujours exactement $k$ membres, et il est un **ensemble canonique** (trie) : l'ordre d'elimination ne fait pas partie du resultat. C'est la meme convention que celle du carnet jumeau SC-10 (manipulation strategique multi-vainqueurs).

**Complexite** : $O(n \cdot m \cdot k)$ -- polynomial.


In [2]:
// STV au quota de Droop : election puis elimination
using System;
using System.Collections.Generic;
using System.Linq;

public static class Sc09Stv
{
    ///<summary>
    /// STV au quota de Droop, variante par election puis elimination.
    /// Renvoie le comite elu, trie (ensemble canonique), de cardinal exactement k.
    ///
    /// Algorithme :
    ///   1. Si un candidat a >= quota, elire le plus haut + transfert fractionnaire du surplus.
    ///   2. Sinon, eliminer le plus bas + transfert des voix de l'elimine vers leur
    ///      prochain prefere vivant non elu.
    /// L'ordre est IMPORTANT : eliminer d'abord ferait perdre les candidats deja au quota.
    ///
    /// Convention : quota de Droop ; transfert du surplus a l'election par fraction
    /// (methode dite Gregory, 1880 -- Hare, 1857, designe le scrutin lui-meme).
    ///
    /// Convention de fin de scrutin : l'elimination cesse des qu'il ne reste pas assez
    /// de survivants pour occuper les sieges encore vacants, et le comite est complete
    /// par les SURVIVANTS (votes decroissants, puis ordre alphabétique). Un candidat
    /// elimine ne peut pas etre reintegre : tous les elimines sont a score nul, les
    /// classer entre eux reviendrait a les departager alphabetiquement au lieu de les
    /// classer par popularite. Le comite rendu a donc toujours exactement k membres,
    /// et l'ordre d'elimination ne fait pas partie du resultat.
    /// </summary>
    public static List<string> Stv(List<string[]> R, int k)
    {
        int n = R.Count;
        var cands = Sc09.AllCands(R);
        if (k > cands.Count)
            throw new ArgumentException($"k={k} > m={cands.Count} : pas assez de candidats pour un comite de taille k");

        int quota = n / (k + 1) + 1;   // quota de Droop
        var elus = new List<string>();
        var vivants = new SortedSet<string>(cands, StringComparer.Ordinal);
        var voix = cands.ToDictionary(c => c, _ => 0.0);

        // Voix initiales : chaque votant attribue son suffrage a son premier prefere
        foreach (var r in R)
        {
            foreach (var c in r)
            {
                if (vivants.Contains(c)) { voix[c] += 1.0; break; }
            }
        }

        while (elus.Count < k)
        {
            // ETAPE 1 : si un candidat est au quota, elire le plus haut (priorite a l'election)
            var auQuota = vivants.Where(c => voix[c] >= quota)
                                 .OrderByDescending(c => voix[c])
                                 .ThenBy(c => c, StringComparer.Ordinal)
                                 .ToList();
            if (auQuota.Count > 0)
            {
                string elu = auQuota[0];
                elus.Add(elu);
                vivants.Remove(elu);

                // Transfert fractionnaire du surplus : chaque votant transfere (voix - q)/voix
                // au prochain prefere vivant
                double total = voix[elu];
                double surplus = total - quota;
                if (surplus > 0)
                {
                    foreach (var r in R)
                    {
                        bool pastElu = false;
                        foreach (var c in r)
                        {
                            if (c == elu) { pastElu = true; continue; }   // skip the elu itself
                            if (pastElu && vivants.Contains(c) && !elus.Contains(c))
                            {
                                voix[c] += surplus / total;   // next alive, transfer surplus fraction
                                break;
                            }
                        }
                    }
                }
                voix[elu] = quota;   // gel
                continue;
            }

            // ETAPE 2 : aucun candidat au quota, eliminer le plus bas -- mais seulement
            // tant qu'il reste assez de survivants pour occuper les sieges vacants.
            if (vivants.Count == 0 || vivants.Count <= k - elus.Count) break;

            string elim = vivants.OrderBy(c => voix[c])
                                 .ThenBy(c => c, StringComparer.Ordinal)
                                 .First();

            // Porteur courant de chaque votant AVANT le retrait de l'elimine.
            var porteurs = new List<string>();
            foreach (var r in R)
            {
                foreach (var c in r)
                {
                    if (vivants.Contains(c) || elus.Contains(c)) { porteurs.Add(c); break; }
                }
            }

            vivants.Remove(elim);

            // Transfert des voix de l'elimine : SEULS les votants dont le porteur
            // courant est l'elimine transferent, vers leur prochain prefere vivant.
            for (int j = 0; j < R.Count; j++)
            {
                if (porteurs[j] != elim) continue;
                foreach (var c in R[j])
                {
                    if (vivants.Contains(c) && !elus.Contains(c)) { voix[c] += 1.0; break; }
                }
            }
            voix[elim] = 0.0;
        }

        // COMPLETION par les survivants (jamais parmi les elimines, tous a score nul).
        if (elus.Count < k)
        {
            var survivants = vivants.OrderByDescending(c => voix[c])
                                    .ThenBy(c => c, StringComparer.Ordinal)
                                    .ToList();
            elus.AddRange(survivants.Take(k - elus.Count));
        }

        if (elus.Count != k)
            throw new InvalidOperationException("STV : le comite rendu doit avoir exactement k membres");

        return elus.OrderBy(c => c, StringComparer.Ordinal).ToList();
    }
}

var elusStv = Sc09Stv.Stv(Sc09.R, Sc09.K);
Console.WriteLine($"STV (Hare) : {Sc09.Fmt(elusStv)}");
Console.WriteLine($"Droop quota : q = {Sc09.N / (Sc09.K + 1) + 1}");
foreach (var c in elusStv)
{
    var rs = Sc09.R.Select(r => Sc09.Rang(r, c)).OrderBy(x => x).ToList();
    Console.WriteLine($"  {c} : rangs obtenus = [{string.Join(", ", rs)}], median = {rs[rs.Count / 2]}");
}


STV (Hare) : ['a', 'c']


Droop quota : q = 2


  a : rangs obtenus = [1, 1, 2, 3, 4], median = 2


  c : rangs obtenus = [1, 2, 3, 3, 4], median = 3


## 3. Monroe (1995) : representants equilibres, NP-dur

La regle de Monroe cherche un comite $W$ ou chaque gagnant $w \in W$
represente un nombre **equilibre** de votants : les $n$ votants sont
partitionnes en $k$ groupes de tailles dans $\{\lfloor n/k \rfloor, \lceil n/k \rceil\}$, et chaque
groupe est assigne a un candidat distinct de $W$. La cle : un votant
peut etre assigne a un candidat **sous-optimal** dans $W$, pourvu que
la taille du groupe soit respectee. C'est cette affectation equilibree
qui distingue Monroe de Chamberlin-Courant (CC) : CC assigne chaque
votant a son prefere dans $W$, sans contrainte d'equilibrage ; Monroe
force l'equilibrage, parfois au prix d'une satisfaction individuelle
inferieure (Borda par votant).

**Definitions formelles** :

- Une **affectation equilibree** $\sigma : N \to W$ est une fonction telle
  que pour tout $c \in W$, $|\sigma^{-1}(c)| \in \{\lfloor n/k \rfloor, \lceil n/k \rceil\}$.
- Un comite $W$ est **Monroe-realisable** ssi il existe une affectation
  equilibree $\sigma : R \to W$. La condition n'est pas triviale :
  pour certains $W$ et certaines distributions de preferences, aucune
  repartition equilibree n'est possible. Ici on adopte la **version
  stricte** (1995, formulation originelle), sans contrainte d'envy --
  un votant peut etre assigne a un candidat sous-optimal pourvu que
  la taille du groupe soit respectee.
- Le comite Monroe est celui qui maximise la satisfaction Borda
  $\sum_i (m - \text{rang}(R_i, \sigma(i)))$ parmi les couples
  $(W, \sigma)$ realisables. Le score est calcule sur l'affectation
  assignee, **PAS** sur le best-in-W (ce serait CC).

**Piege classique -- confondre Monroe et Chamberlin-Courant.** Une
implementation qui assigne chaque votant a son prefere dans $W$ puis
verifie les quotas rend en realite **Chamberlin-Courant** : l'affectation
top-choice ne respecte les bornes $[\lfloor n/k \rfloor, \lceil n/k \rceil]$
que dans les cas triviaux. Le discriminant executable entre les deux
regles est exactement $|\sigma^{-1}(c)| \in \{\lfloor n/k \rfloor, \lceil n/k \rceil\}$
pour tout $c \in W$ -- sur l'affectation construite, pas sur le rang du
prefere.

**Complexite** : **NP-dur** (Monroe 1995, Procaccia et al. 2008). La
recherche par enumeration conjointe (k-comites parmi m, backtracking
sur les affectations equilibrees) est $O(C(m, k) \cdot k^n)$ en worst
case, mais l'elaguage par quota (chaque candidat ne peut accepter que
$\lceil n/k \rceil$ votants) rend le worst case rare pour $n \le 8$.
Les algorithmes appropries sont MIP (PLNE en nombres entiers) et
algorithmes d'approximation avec garantie $1 - 1/e$ (Skowron et al.).

**Implementation** : enumeration conjointe comite/affectation. Pour
chaque $W$, backtrack sur les affectations equilibrees (quotas
[lo, hi] respectes), garder la max-Borda (sur $\sigma$, pas best-in-W).


In [3]:
// Monroe (1995 stricte) : enumeration conjointe comite + backtracking equilibre
using System;
using System.Collections.Generic;
using System.Linq;

public static class Sc09Monroe
{
    /// <summary>
    /// Pour chaque votant i, retourne le candidat de W le mieux place dans son bulletin.
    /// Si W est vide, retourne null pour le votant.
    /// C'est la definition Chamberlin-Courant (top-choice dans W) --
    /// a NE PAS utiliser pour Monroe, qui exige une affectation equilibree.
    /// </summary>
    public static List<string> Assignation(List<string[]> R, List<string> W)
    {
        if (W.Count == 0) return Enumerable.Repeat<string>(null, R.Count).ToList();
        var sigma = new List<string>();
        foreach (var r in R)
            sigma.Add(W.OrderBy(c => Sc09.Rang(r, c)).First());
        return sigma;
    }

    /// <summary>
    /// Regle de Monroe (1995, stricte) : comite + affectation equilibree.
    /// Renvoie (W, sigma) ; W est null si aucun W realisable.
    ///
    /// W de taille k est Monroe-realise s'il existe une affectation
    /// sigma : R -> W telle que, pour tout c in W, |sigma^-1(c)| appartient
    /// a [lo, hi] = [floor(n/k), ceil(n/k)] (quota equilibre par elu).
    ///
    /// L'optimisation maximise la satisfaction Borda sum_i (m - rang(R_i, sigma_i))
    /// parmi les couples (W, sigma) realisables. Le score est calcule sur
    /// l'affectation assignee, PAS sur le best-in-W (ce serait CC).
    ///
    /// Implementation : enumeration conjointe. Pour chaque W (k parmi m),
    /// backtracking sur les affectations equilibrees (quotas [lo, hi])
    /// avec elaguage (chaque candidat ne peut accepter que hi votants).
    /// Pour chaque W, on garde la sigma max-Borda. Parmi tous les W,
    /// on garde le (W, sigma) avec le plus haut score Borda.
    ///
    /// Complexite : O(C(m, k) * k^n) en worst case, elaguage par quota
    /// rend le worst case rare pour n <= 8.
    /// </summary>
    public static (List<string> W, List<string> sigma) Monroe(List<string[]> R, int k, int? mOpt = null)
    {
        int n = R.Count;
        var cands = Sc09.AllCands(R);
        int m = mOpt ?? R[0].Length;
        int lo = n / k;
        int hi = (n + k - 1) / k;

        (List<string> sigma, int score) MeilleureAffectation(string[] W)
        {
            var quotas = W.ToDictionary(c => c, _ => 0);
            var sigma = new string[n];
            List<string> best = null;
            int bestScore = -1;

            void Backtrack(int i)
            {
                if (i == n)
                {
                    if (W.All(c => quotas[c] >= lo))
                    {
                        int score = 0;
                        for (int j = 0; j < n; j++) score += m - Sc09.Rang(R[j], sigma[j]);
                        if (score > bestScore)
                        {
                            bestScore = score;
                            best = sigma.ToList();
                        }
                    }
                    return;
                }
                foreach (var c in W)
                {
                    if (quotas[c] < hi)
                    {
                        sigma[i] = c;
                        quotas[c] += 1;
                        Backtrack(i + 1);
                        quotas[c] -= 1;
                        sigma[i] = null;
                    }
                }
            }

            Backtrack(0);
            return (best, bestScore);
        }

        List<string> meilleurW = null;
        List<string> meilleurSigma = null;
        int meilleurScore = -1;

        foreach (var W in Sc09.Combinations(cands, k))
        {
            var (sigma, score) = MeilleureAffectation(W.ToArray());
            if (sigma != null && score > meilleurScore)
            {
                meilleurScore = score;
                meilleurW = W.ToList();
                meilleurSigma = sigma;
            }
        }

        return (meilleurW, meilleurSigma);
    }

    /// <summary>
    /// Renvoie (quotas_ok, dict_quota) pour une affectation sigma et un comite W.
    /// quotas_ok = True ssi |sigma^-1(c)| in [lo, hi] pour tout c in W.
    /// </summary>
    public static (bool ok, Dictionary<string, int> quotas) VerifierQuotas(List<string> sigma, List<string> W, int n, int k)
    {
        int lo = n / k;
        int hi = (n + k - 1) / k;
        var quotas = W.ToDictionary(c => c, _ => 0);
        if (sigma != null)
        {
            foreach (var s in sigma)
                if (s != null && quotas.ContainsKey(s)) quotas[s] += 1;
        }
        bool ok = W.All(c => quotas[c] >= lo && quotas[c] <= hi);
        return (ok, quotas);
    }
}

var Rfil = Sc09.R;
int Nfil = Sc09.N;
int mfil = Sc09.M;
int Kfil = Sc09.K;

var (WMonroe, sigmaMonroe) = Sc09Monroe.Monroe(Rfil, Kfil, mfil);
if (WMonroe == null)
{
    Console.WriteLine("Aucun comite Monroe realisable sur ce profil.");
}
else
{
    Console.WriteLine($"Monroe : W = {Sc09.Fmt(WMonroe)}");
    Console.WriteLine($"Quotas : floor = {Nfil / Kfil}, ceil = {(Nfil + Kfil - 1) / Kfil}");
    var (quotasOk, quotas) = Sc09Monroe.VerifierQuotas(sigmaMonroe, WMonroe, Nfil, Kfil);
    Console.WriteLine($"Verification quotas equilibres : {quotasOk} (details : " +
        string.Join(", ", quotas.OrderBy(kv => kv.Key, StringComparer.Ordinal).Select(kv => $"{kv.Key}: {kv.Value}")) + ")");
    foreach (var c in WMonroe)
    {
        var representes = sigmaMonroe.Select((s, i) => (s, i))
                                    .Where(t => t.s == c)
                                    .Select(t => t.i + 1)
                                    .ToList();
        Console.WriteLine($"  {c} represente les votants [{string.Join(", ", representes)}] (n = {representes.Count})");
    }
    int scoreAssignee = Enumerable.Range(0, Nfil).Sum(i => mfil - Sc09.Rang(Rfil[i], sigmaMonroe[i]));
    Console.WriteLine($"Satisfaction Borda sur affectation assignee : {scoreAssignee}");
    Console.WriteLine("Satisfaction Borda best-in-W (CC-like, pour comparaison) : " +
        Rfil.Sum(r => Sc09.ScoreBorda(r, WMonroe, mfil)));
}


Monroe : W = ['a', 'c']


Quotas : floor = 2, ceil = 3


Verification quotas equilibres : True (details : a: 3, c: 2)


  a represente les votants [1, 2, 5] (n = 3)


  c represente les votants [3, 4] (n = 2)


Satisfaction Borda sur affectation assignee : 13


Satisfaction Borda best-in-W (CC-like, pour comparaison) : 13


## 4. Chamberlin-Courant (1983) : assignation Borda optimale, NP-dur

La regle de Chamberlin-Courant (CC) cherche un comite $W$ qui maximise la **satisfaction Borda agregée** : $\sum_{i=1}^n u_i(W)$ ou $u_i(W) = \max_{c \in W} (M - rang(R_i, c))$ est le score Borda du candidat le mieux place dans $W$ pour le votant $i$.

**Complexite** : **NP-dur** sous Borda ou approval (procaccia et al. 2008, Lu & Boutilier 2011). L'enumeration brute-force est $O(C(m, k) \cdot n \cdot k)$.

**Implementation** :

- Enumeration directe par `Combinations` (combinaisons de taille k) pour les profils jouets ($m \le 5, k \le 3$) ;
- Z3 via `Microsoft.Z3` : `Optimize` avec variables binaires $x_{jc} \in \{0, 1\}$ (votant $j$ attribue au candidat $c$) + $\sum_c x_{jc} = 1$ (chaque votant a un representant) + $\sum_c y_c = k$ (comite de taille $k$) + $y_c \ge x_{jc}$ (on ne peut attribuer qu'a un elu). Maximiser $\sum_{j, c} \text{borda}_{jc} \cdot x_{jc}$.


In [4]:
// Chamberlin-Courant : enumeration brute-force des comites
using System;
using System.Collections.Generic;
using System.Linq;

public static class Sc09Cc
{
    /// <summary>
    /// Chamberlin-Courant : enumeration directe (brute-force) sur les comites.
    /// Renvoie (W optimal, score Borda agrege maximal).
    /// m = nombre de candidats (defaut null : infere de R[0]).
    /// </summary>
    public static (List<string> W, int score) CcBrute(List<string[]> R, int k, int? mOpt = null)
    {
        int m = mOpt ?? R[0].Length;
        var cands = Sc09.AllCands(R);
        List<string> meilleur = null;
        int meilleurScore = -1;

        foreach (var W in Sc09.Combinations(cands, k))
        {
            int score = R.Sum(r => Sc09.ScoreBorda(r, W, m));
            if (score > meilleurScore)
            {
                meilleurScore = score;
                meilleur = W;
            }
        }
        return (meilleur, meilleurScore);
    }
}

var (Wcc, scoreCc) = Sc09Cc.CcBrute(Sc09.R, Sc09.K, Sc09.M);
Console.WriteLine($"CC brute : W = {Sc09.Fmt(Wcc)}, score Borda agrege = {scoreCc}");
for (int i = 0; i < Sc09.R.Count; i++)
{
    string bestInW = Wcc.OrderBy(c => Sc09.Rang(Sc09.R[i], c)).First();
    Console.WriteLine($"  v{i + 1} : prefere {bestInW} dans W (rang {Sc09.Rang(Sc09.R[i], bestInW)}, " +
        $"Borda {Sc09.ScoreBorda(Sc09.R[i], Wcc, Sc09.M)})");
}


CC brute : W = ['a', 'c'], score Borda agrege = 13


  v1 : prefere a dans W (rang 1, Borda 3)


  v2 : prefere a dans W (rang 1, Borda 3)


  v3 : prefere c dans W (rang 2, Borda 2)


  v4 : prefere c dans W (rang 1, Borda 3)


  v5 : prefere a dans W (rang 2, Borda 2)


## 5. Z3 confrontation : solveur SOTA pour Monroe et Chamberlin-Courant

On formule les deux regles NP-dures en **programmation lineaire en nombres entiers** resolue par `Microsoft.Z3.Optimize`. La formulation commune encode :

- $y_c \in \{0, 1\}$ : candidat $c$ elu ?
- $x_{jc} \in \{0, 1\}$ : votant $j$ attribue au candidat $c$ ?
- $\sum_c y_c = k$ : comite de taille $k$.
- $\sum_c x_{jc} = 1$ : chaque votant attribue a un unique candidat.
- $x_{jc} \le y_c$ : un votant ne peut etre attribue qu'a un candidat elu.
- Maximiser $\sum_{j, c} \text{borda}_{jc} \cdot x_{jc}$.

Pour Monroe, la formulation ajoute **une seule contrainte** -- l'equilibre des charges :
$y_c = 1 \Rightarrow \lfloor n/k \rfloor \le \sum_j x_{jc} \le \lceil n/k \rceil$,
c'est-a-dire que chaque candidat elu doit representer entre $\lfloor n/k \rfloor$ et
$\lceil n/k \rceil$ votants. C'est l'unique difference de formulation entre les deux
regles ; le profil discriminant en fin de section montre qu'elle change le comite optimal.

**Interet de la confrontation** : sur des profils jouets ($n=5, m=4, k=2$), l'enumeration directe
(6 comites) est triviale mais permet de **verifier que Z3 produit le meme optimum** qu'une
recherche exhaustive -- y compris pour Monroe, dont l'affectation equilibree est verifiee
quota par quota.


In [5]:
// Z3 Optimize : Chamberlin-Courant et Monroe (contrainte d'equilibre des charges)
#r "nuget: Microsoft.Z3, 4.12.2"
#load "../../SymbolicAI/SMT/Z3-API/Z3NativeLoader.cs"
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.Linq;

public static class Sc09Random
{
    public static string[] Sample(Random rng, IEnumerable<string> source)
    {
        var items = source.ToList();
        for (int i = items.Count - 1; i > 0; i--)
        {
            int j = rng.Next(i + 1);
            (items[i], items[j]) = (items[j], items[i]);
        }
        return items.ToArray();
    }
}

public static class Sc09Z3
{
    private static int LongValue(Model model, IntExpr expression)
        => (int)((IntNum)model.Evaluate(expression, true)).Int64;

    // Somme lineaire large : MkMul rend des ArithExpr, les termes purs sont IntExpr
    // (variance ok, IEnumerable<out T>). MkAdd rend ArithExpr, cast IntExpr valide
    // car tous les termes sont entiers.
    private static IntExpr Sum(Context ctx, IEnumerable<ArithExpr> terms)
        => (IntExpr)ctx.MkAdd(terms.ToArray());

    // Equivalent lineaire exact des pseudo-Booleens MkPBEq/MkPBLe : chaque variable
    // passee ici est deja contrainte a {0, 1} par le corps de CcZ3/MonroeZ3
    // (MkGe 0 / MkLe 1), donc somme lineaire == nombre de variables a 1.
    // Meme formulation que le source Python (Sum([...]) == k, pas de PbEq natif).
    private static BoolExpr PbEq(Context ctx, IEnumerable<IntExpr> variables, int target)
        => ctx.MkEq(Sum(ctx, variables), ctx.MkInt(target));

    private static BoolExpr PbLe(Context ctx, IEnumerable<IntExpr> variables, int target)
        => ctx.MkLe(Sum(ctx, variables), ctx.MkInt(target));

    /// <summary>
    /// Chamberlin-Courant par Z3 Optimize. Variables :
    /// - y_c in {0, 1} : candidat c elu
    /// - x_jc in {0, 1} : votant j attribue au candidat c
    /// Maximise sum_j sum_c borda(R_j, c) * x_jc.
    /// </summary>
    public static (List<string> W, List<string> sigma, int score) CcZ3(List<string[]> R, int k, int? mOpt = null)
    {
        int m = mOpt ?? R[0].Length;
        int n = R.Count;
        var cands = Sc09.AllCands(R);
        using var ctx = new Context();
        var opt = ctx.MkOptimize();
        var y = cands.ToDictionary(c => c, c => ctx.MkIntConst("y_" + c));
        var x = new Dictionary<(int j, string c), IntExpr>();

        foreach (var c in cands)
        {
            opt.Add(ctx.MkGe(y[c], ctx.MkInt(0)));
            opt.Add(ctx.MkLe(y[c], ctx.MkInt(1)));
        }
        for (int j = 0; j < n; j++)
        {
            foreach (var c in cands)
            {
                var variable = ctx.MkIntConst($"x_{j}_{c}");
                x[(j, c)] = variable;
                opt.Add(ctx.MkGe(variable, ctx.MkInt(0)));
                opt.Add(ctx.MkLe(variable, ctx.MkInt(1)));
            }
            opt.Add(PbEq(ctx, cands.Select(c => x[(j, c)]), 1));
        }
        opt.Add(PbEq(ctx, cands.Select(c => y[c]), k));

        foreach (var c in cands)
            for (int j = 0; j < n; j++)
                opt.Add(ctx.MkLe(x[(j, c)], y[c]));

        var borda = new Dictionary<(int j, string c), int>();
        for (int j = 0; j < n; j++)
            foreach (var c in cands)
                borda[(j, c)] = Sc09.ScoreBorda(R[j], new[] { c }, m);

        IntExpr scoreTotal = Sum(ctx, borda.Select(pair => ctx.MkMul(ctx.MkInt(pair.Value), x[pair.Key])));
        opt.MkMaximize(scoreTotal);

        if (opt.Check() != Status.SATISFIABLE) return (null, null, -1);
        var model = opt.Model;
        var W = cands.Where(c => LongValue(model, y[c]) == 1).ToList();
        var sigma = new List<string>();
        for (int j = 0; j < n; j++)
            sigma.Add(cands.First(c => LongValue(model, x[(j, c)]) == 1));

        int score = R.Sum(r => Sc09.ScoreBorda(r, W, m));
        return (W, sigma, score);
    }

    /// <summary>
    /// Monroe (strict, 1995) par Z3 Optimize : formulation CC + contrainte
    /// d'equilibre -- chaque candidat elu doit representer entre floor(n/k) et
    /// ceil(n/k) votants. C'est l'unique difference de formulation avec CC
    /// (absence de la contrainte => la regle est CC, pas Monroe).
    /// </summary>
    public static (List<string> W, List<string> sigma, int score) MonroeZ3(List<string[]> R, int k, int? mOpt = null)
    {
        int m = mOpt ?? R[0].Length;
        int n = R.Count;
        var cands = Sc09.AllCands(R);
        int lo = n / k;
        int hi = (n + k - 1) / k;

        using var ctx = new Context();
        var opt = ctx.MkOptimize();
        var y = cands.ToDictionary(c => c, c => ctx.MkIntConst("y_" + c));
        var x = new Dictionary<(int j, string c), IntExpr>();

        foreach (var c in cands)
        {
            opt.Add(ctx.MkGe(y[c], ctx.MkInt(0)));
            opt.Add(ctx.MkLe(y[c], ctx.MkInt(1)));
        }
        for (int j = 0; j < n; j++)
        {
            foreach (var c in cands)
            {
                var variable = ctx.MkIntConst($"x_{j}_{c}");
                x[(j, c)] = variable;
                opt.Add(ctx.MkGe(variable, ctx.MkInt(0)));
                opt.Add(ctx.MkLe(variable, ctx.MkInt(1)));
            }
            opt.Add(PbEq(ctx, cands.Select(c => x[(j, c)]), 1));
        }
        opt.Add(PbEq(ctx, cands.Select(c => y[c]), k));
        foreach (var c in cands)
            for (int j = 0; j < n; j++)
                opt.Add(ctx.MkLe(x[(j, c)], y[c]));

        foreach (var c in cands)
        {
            var charge = Sum(ctx, Enumerable.Range(0, n).Select(j => x[(j, c)]));
            opt.Add(ctx.MkImplies(ctx.MkEq(y[c], ctx.MkInt(1)), ctx.MkGe(charge, ctx.MkInt(lo))));
            opt.Add(PbLe(ctx, Enumerable.Range(0, n).Select(j => x[(j, c)]), hi));
        }

        var borda = new Dictionary<(int j, string c), int>();
        for (int j = 0; j < n; j++)
            foreach (var c in cands)
                borda[(j, c)] = Sc09.ScoreBorda(R[j], new[] { c }, m);

        IntExpr scoreTotal = Sum(ctx, borda.Select(pair => ctx.MkMul(ctx.MkInt(pair.Value), x[pair.Key])));
        opt.MkMaximize(scoreTotal);

        if (opt.Check() != Status.SATISFIABLE) return (null, null, -1);
        var model = opt.Model;
        var W = cands.Where(c => LongValue(model, y[c]) == 1).ToList();
        var sigma = new List<string>();
        for (int j = 0; j < n; j++)
            sigma.Add(cands.First(c => LongValue(model, x[(j, c)]) == 1));

        int score = Enumerable.Range(0, n).Sum(j => m - Sc09.Rang(R[j], sigma[j]));
        return (W, sigma, score);
    }
}

var (Wz3, sigmaZ3, scoreZ3) = Sc09Z3.CcZ3(Sc09.R, Sc09.K, Sc09.M);
var (WccBrute, scoreCcBrute) = Sc09Cc.CcBrute(Sc09.R, Sc09.K, Sc09.M);
Console.WriteLine($"CC Z3 Optimize : W = {Sc09.Fmt(Wz3)}, score = {scoreZ3}");
Console.WriteLine($"CC brute       : W = {Sc09.Fmt(WccBrute)}, score = {scoreCcBrute}");
bool verdictCc = Wz3.SequenceEqual(WccBrute) && scoreZ3 == scoreCcBrute;
Console.WriteLine($"Verdict confrontation : {(verdictCc ? "OK memes optima" : "DIVERGENCE")}");

var rngZ3 = new Random(20261005);
Console.WriteLine("\nTest sur 5 profils aleatoires (n in [4,5], m in [3,4], k in [2,3]) :");
int nbConcordancesCc = 0;
for (int t = 0; t < 5; t++)
{
    int nt = rngZ3.Next(4, 6);
    int mt = rngZ3.Next(3, 5);
    int kt = rngZ3.Next(2, Math.Min(3, mt) + 1);
    var candsT = Enumerable.Range(0, mt).Select(j => ((char)('a' + j)).ToString()).ToList();
    var Rt = Enumerable.Range(0, nt).Select(_ => Sc09Random.Sample(rngZ3, candsT)).ToList();
    var (WbruteT, sBruteT) = Sc09Cc.CcBrute(Rt, kt, mt);
    var (Wz3T, _, sZ3T) = Sc09Z3.CcZ3(Rt, kt, mt);
    // Concordance des deux voies au niveau de l'OPTIMUM : la force brute enumere
    // tous les comites, son score est l'optimum exact ; Z3 doit l'atteindre.
    // En cas d'ex aequo (plusieurs comites optimaux), Z3 peut rendre un autre
    // representant optimal -- variabilite de representant anticipee par le source.
    bool okT = sZ3T == sBruteT;
    if (okT) nbConcordancesCc++;
    Console.WriteLine($"  t={t}: n={nt}, m={mt}, k={kt}  brute={Sc09.Fmt(WbruteT)} (s={sBruteT})  " +
        $"Z3={Sc09.Fmt(Wz3T)} (s={sZ3T})  {(Wz3T.SequenceEqual(WbruteT) ? (okT ? "OK" : "DIVERGENCE") : (okT ? "OK (ex aequo)" : "DIVERGENCE"))}");
}
Console.WriteLine($"Concordance brute/Z3 : {nbConcordancesCc}/5 ({100.0 * nbConcordancesCc / 5:F0} %)");
if (nbConcordancesCc != 5) throw new InvalidOperationException("Z3 doit retrouver l'optimum de CC sur les cinq profils");

var RTemoinM = Enumerable.Repeat(new[] { "a", "b", "c" }, 4).ToList();
var (WMzT, sigMzT, scMzT) = Sc09Z3.MonroeZ3(RTemoinM, 2, 3);
var (quotasOkT, quotasT) = Sc09Monroe.VerifierQuotas(sigMzT, WMzT, 4, 2);
Console.WriteLine($"\nTemoin monroe_z3 sur [('a','b','c')]*4, k=2 : W = {Sc09.Fmt(WMzT)}, " +
    $"sigma = {Sc09.Fmt(sigMzT)}, score = {scMzT}");
Console.WriteLine("Quotas par elu : " + string.Join(", ", quotasT.OrderBy(kv => kv.Key).Select(kv => $"{kv.Key}: {kv.Value}")) +
    " -- equilibre [floor, ceil] = [2, 2] respecte : " + quotasOkT);
if (!quotasOkT) throw new InvalidOperationException("Le temoin Monroe doit respecter les quotas [2,2]");

var RDisc = new List<string[]>
{
    new[] { "c", "a", "d", "b" },
    new[] { "d", "a", "b", "c" },
    new[] { "d", "a", "b", "c" },
    new[] { "d", "a", "b", "c" },
    new[] { "d", "a", "b", "c" },
    new[] { "d", "a", "c", "b" },
};
int kDisc = 2;
int mDisc = 4;
int nDisc = RDisc.Count;

var (WMBrute, sigMBrute) = Sc09Monroe.Monroe(RDisc, kDisc, mDisc);
int scoreMBrute = Enumerable.Range(0, nDisc).Sum(i => mDisc - Sc09.Rang(RDisc[i], sigMBrute[i]));
var (WCCBrute, scoreCCBrute) = Sc09Cc.CcBrute(RDisc, kDisc, mDisc);
var (WMzD, sigMzD, scoreMzD) = Sc09Z3.MonroeZ3(RDisc, kDisc, mDisc);
var (WCCzD, sigCCzD, scoreCCzD) = Sc09Z3.CcZ3(RDisc, kDisc, mDisc);

bool okM = WMzD.SequenceEqual(WMBrute) && scoreMzD == scoreMBrute;
bool okCC = WCCzD.SequenceEqual(WCCBrute) && scoreCCzD == scoreCCBrute;
var (quotasOkD, quotasD) = Sc09Monroe.VerifierQuotas(sigMzD, WMzD, nDisc, kDisc);
var chargesCC = sigCCzD.GroupBy(c => c).ToDictionary(g => g.Key, g => g.Count());

Console.WriteLine("\nProfil discriminant (n=6, m=4, k=2) : " +
    "1 x (c>a>d>b), 4 x (d>a>b>c), 1 x (d>a>c>b)");
Console.WriteLine("Quota Monroe par elu : [floor(6/2), ceil(6/2)] = [3, 3]");
Console.WriteLine();
Console.WriteLine($"{"Regle",-8}{"Comite",-12}{"Score",-7}Detail");
Console.WriteLine(new string('-', 66));
Console.WriteLine($"{"Monroe",-8}{Sc09.Fmt(WMBrute.OrderBy(c => c).ToList()),-12}{scoreMBrute,-7}brute : sigma = {Sc09.Fmt(sigMBrute)}");
Console.WriteLine($"{"Monroe",-8}{Sc09.Fmt(WMzD.OrderBy(c => c).ToList()),-12}{scoreMzD,-7}Z3    : sigma = {Sc09.Fmt(sigMzD)} (quotas {string.Join(", ", quotasD.OrderBy(kv => kv.Key).Select(kv => $"{kv.Key}: {kv.Value}"))})");
Console.WriteLine($"{"CC",-8}{Sc09.Fmt(WCCBrute.OrderBy(c => c).ToList()),-12}{scoreCCBrute,-7}brute : best-in-W par votant");
Console.WriteLine($"{"CC",-8}{Sc09.Fmt(WCCzD.OrderBy(c => c).ToList()),-12}{scoreCCzD,-7}Z3    : charges par elu {string.Join(", ", chargesCC.OrderBy(kv => kv.Key).Select(kv => $"{kv.Key}: {kv.Value}"))}");
Console.WriteLine();
Console.WriteLine($"Accord Z3 / brute-force : Monroe {(okM ? "OK" : "DIVERGENCE")}, CC {(okCC ? "OK" : "DIVERGENCE")}");
Console.WriteLine($"Les deux regles elisent des comites differents : Monroe {Sc09.Fmt(WMzD.OrderBy(c => c).ToList())} vs CC {Sc09.Fmt(WCCzD.OrderBy(c => c).ToList())}.");
Console.WriteLine($"CC satisfait le votant 1 avec c (charges non equilibrees {string.Join(", ", chargesCC.OrderBy(kv => kv.Key).Select(kv => $"{kv.Key}: {kv.Value}"))}) ;");
Console.WriteLine("Monroe, contraint par [3, 3], elue a a la place de c : chaque votant garde");
Console.WriteLine("un representant de rang <= 2, et le comite de CC ne vaudrait que 13 en Monroe.");
if (!okM || !okCC) throw new InvalidOperationException("Z3 doit retrouver l'optimum des deux regles");
if (!quotasOkD) throw new InvalidOperationException("L'affectation de monroe_z3 doit respecter les quotas d'equilibre");
if (WMzD.OrderBy(c => c).SequenceEqual(WCCzD.OrderBy(c => c)))
    throw new InvalidOperationException("Le profil doit discriminer Monroe et CC");

Installed Packages Microsoft.Z3, 4.12.2

CC Z3 Optimize : W = ['a', 'c'], score = 13


CC brute       : W = ['a', 'c'], score = 13


Verdict confrontation : OK memes optima



Test sur 5 profils aleatoires (n in [4,5], m in [3,4], k in [2,3]) :


  t=0: n=5, m=4, k=3  brute=['a', 'b', 'c'] (s=15)  Z3=['a', 'b', 'c'] (s=15)  OK


... (16 outputs left out)

Les deux regles elisent des comites differents : Monroe ['a', 'd'] vs CC ['c', 'd'].


CC satisfait le votant 1 avec c (charges non equilibrees c: 1, d: 5) ;


Monroe, contraint par [3, 3], elue a a la place de c : chaque votant garde


un representant de rang <= 2, et le comite de CC ne vaudrait que 13 en Monroe.


## 6. Experimentation : fil rouge + profils aleatoires

Comparons les trois regles sur le fil rouge, puis sur un echantillon aleatoire de profils (mesure du taux d'accord des comites elus).

**Lecture attendue** :

- Sur les profils structures (fil rouge, blocs symetriques), les trois regles elisent le meme comite. Sur des profils aleatoires uniformes, le STV s'ecarte plus souvent des deux regles d'affectation : le transfert de voix (surplus puis eliminations successives) ne construit pas d'affectation equilibree, la ou Monroe et CC optimisent une affectation. L'ecart reste neanmoins plus faible qu'il n'y parait : quand les comites compares ont le meme cardinal, les trois regles convergent souvent.
- Z3 doit toujours retourner l'optimum exact sur les profils jouets (limite de complexite CJ10e11 pour les profils $n>10$).
- Sur les profils aleatoires, le taux d'accord CC vs Monroe depend de la structure des preferences.
- **Taux propres a ce carnet** : `System.Random` (graines 20261005/20261006) ne tire pas les memes profils que le `random` de Python a graine egale — les taux affiches ci-dessous (65 % / 85 % / 55 % sur 20 profils) sont ceux des tirages de CE carnet C#. Le jumeau Python, sur SES propres tirages, mesure 75 % / 95 % / 70 %. Meme protocole, memes graines nominales, sequences differentes : comparer les ORDRES DE GRANDEUR, jamais les chiffres au pourcent pres.

In [6]:
// Comparaison STV / Monroe / CC sur le fil rouge, 20 profils aleatoires, temoins et garde STV
using System;
using System.Collections.Generic;
using System.Linq;

var (WmonroeFil, sigmaMonroeFil) = Sc09Monroe.Monroe(Sc09.R, Sc09.K, Sc09.M);
var (WccFil, scoreCcFil) = Sc09Cc.CcBrute(Sc09.R, Sc09.K, Sc09.M);
var (Wz3Fil, sigmaZ3Fil, scoreZ3Fil) = Sc09Z3.CcZ3(Sc09.R, Sc09.K, Sc09.M);

Console.WriteLine("Comparaison sur le fil rouge (n=5, m=4, k=2) :");
Console.WriteLine($"  STV (Hare, election puis elimination) : {Sc09.Fmt(elusStv)}");
Console.WriteLine($"  Monroe (1995 stricte, backtracking equilibre) : W = {Sc09.Fmt(WmonroeFil)}, sigma equilibre");
Console.WriteLine($"  CC brute                : {Sc09.Fmt(WccFil)} (score Borda = {scoreCcFil})");
Console.WriteLine($"  CC Z3 Optimize          : {Sc09.Fmt(Wz3Fil)} (score Borda = {scoreZ3Fil})");

int nProfils = 20;
int accordsStvMonroe = 0;
int accordsMonroeCc = 0;
int accordsStvCc = 0;
var rngExp = new Random(20261005);
for (int t = 0; t < nProfils; t++)
{
    int n = rngExp.Next(4, 7);
    int m = rngExp.Next(3, 6);
    int k = rngExp.Next(2, Math.Min(3, m) + 1);
    var candsT = Enumerable.Range(0, m).Select(j => ((char)('a' + j)).ToString()).ToList();
    var Rt = Enumerable.Range(0, n).Select(_ => Sc09Random.Sample(rngExp, candsT)).ToList();
    var stvT = Sc09Stv.Stv(Rt, k);
    var (WmonroeT, sigmaMonroeT) = Sc09Monroe.Monroe(Rt, k, m);
    var monroeT = WmonroeT == null ? null : WmonroeT.OrderBy(c => c).ToList();
    var (WccT, _) = Sc09Cc.CcBrute(Rt, k, m);
    var ccT = WccT.OrderBy(c => c).ToList();

    if (stvT.SequenceEqual(monroeT)) accordsStvMonroe++;
    if (monroeT.SequenceEqual(ccT)) accordsMonroeCc++;
    if (stvT.SequenceEqual(ccT)) accordsStvCc++;
}

Console.WriteLine($"\nSur {nProfils} profils aleatoires :");
Console.WriteLine($"  STV = Monroe    : {accordsStvMonroe}/{nProfils} ({100.0 * accordsStvMonroe / nProfils:F0}%)");
Console.WriteLine($"  Monroe = CC     : {accordsMonroeCc}/{nProfils} ({100.0 * accordsMonroeCc / nProfils:F0}%)");
Console.WriteLine($"  STV = CC        : {accordsStvCc}/{nProfils} ({100.0 * accordsStvCc / nProfils:F0}%)");

var RTemoin = Enumerable.Repeat(new[] { "a", "b", "c" }, 4).ToList();
var stvTemoin = Sc09Stv.Stv(RTemoin, 2);
var (WmonroeTemoin, sigmaMonroeTemoin) = Sc09Monroe.Monroe(RTemoin, 2, 3);
var (WccTemoin, sCcTemoin) = Sc09Cc.CcBrute(RTemoin, 2, 3);
Console.WriteLine("\nTemoin R=[('a','b','c')]*4, k=2 :");
Console.WriteLine($"  STV    : {Sc09.Fmt(stvTemoin)}  (attendu ['a','b'])");
Console.WriteLine($"  Monroe : W = {Sc09.Fmt(WmonroeTemoin)}, sigma = {Sc09.Fmt(sigmaMonroeTemoin)}  (equilibre 2/2)");
Console.WriteLine($"  CC     : {Sc09.Fmt(WccTemoin)} (Borda {sCcTemoin})");

var R2 = new List<string[]>
{
    new[] { "a", "b", "c" }, new[] { "a", "b", "c" }, new[] { "a", "b", "c" },
    new[] { "b", "c", "a" }, new[] { "b", "c", "a" }, new[] { "b", "c", "a" },
};
Console.WriteLine("\nDeux blocs symetriques : 3x(a>b>c) + 3x(b>c>a), k = 2");
var stv2 = Sc09Stv.Stv(R2, 2);
var (Wm2, sigmaM2) = Sc09Monroe.Monroe(R2, 2, 3);
var (Wcc2, sCc2) = Sc09Cc.CcBrute(R2, 2, 3);
Console.WriteLine($"  STV      : {Sc09.Fmt(stv2)}");
Console.WriteLine($"  Monroe   : W = {Sc09.Fmt(Wm2)}, sigma = {Sc09.Fmt(sigmaM2)}");
Console.WriteLine($"  CC       : {Sc09.Fmt(Wcc2)} (Borda {sCc2})");

var rngGarde = new Random(20261006);
int nComitesVerifies = 0;
for (int t = 0; t < 200; t++)
{
    int nG = rngGarde.Next(4, 7);
    int mG = rngGarde.Next(3, 6);
    int kG = rngGarde.Next(1, Math.Min(3, mG) + 1);
    var candsG = Enumerable.Range(0, mG).Select(j => ((char)('a' + j)).ToString()).ToList();
    var RG = Enumerable.Range(0, nG).Select(_ => Sc09Random.Sample(rngGarde, candsG)).ToList();
    var WG = Sc09Stv.Stv(RG, kG);
    if (WG.Count != kG)
        throw new InvalidOperationException($"STV : cardinal {WG.Count} different de k={kG}");
    if (!WG.SequenceEqual(WG.OrderBy(c => c, StringComparer.Ordinal)))
        throw new InvalidOperationException("STV : le comite n'est pas trie");
    nComitesVerifies++;
}
Console.WriteLine($"\nGARDE STV : {nComitesVerifies} comites verifies sur 200 profils aleatoires " +
    "($n \\in [4,6]$, $m \\in [3,5]$, $k \\in [1,3]$) -- tous de cardinal k, tous tries.");


Comparaison sur le fil rouge (n=5, m=4, k=2) :


  STV (Hare, election puis elimination) : ['a', 'c']


  Monroe (1995 stricte, backtracking equilibre) : W = ['a', 'c'], sigma equilibre


  CC brute                : ['a', 'c'] (score Borda = 13)


  CC Z3 Optimize          : ['a', 'c'] (score Borda = 13)



Sur 20 profils aleatoires :


... (8 outputs left out)

  STV      : ['a', 'b']


  Monroe   : W = ['a', 'b'], sigma = ['a', 'a', 'a', 'b', 'b', 'b']


  CC       : ['a', 'b'] (Borda 12)



GARDE STV : 200 comites verifies sur 200 profils aleatoires ($n \in [4,6]$, $m \in [3,5]$, $k \in [1,3]$) -- tous de cardinal k, tous tries.


## Exercices

Les trois exercices utilisent le fil rouge, les fonctions `Stv`, `Monroe`, `CcBrute`, `CcZ3` et `ScoreBorda` des sections precedentes.


### Exercice 1 -- STV par transfert fractionnaire vs elimination

L'implementation `Stv` de la section 2 utilise une variante simplifiee par elimination successive, pas le transfert fractionnaire strict de Hare. La difference est visible sur les profils ou un candidat depasse largement le quota.

1. Construisez un profil a 6 votants ou le candidat $a$ est premier pour 5 votants et $b$ est premier pour 1. Avec $k = 2$, la variante par elimination doit elire $\{a, ?\}$ ; le transfert fractionnaire stricte peut elire $\{a, b\}$ grace au surplus transfere au candidat en deuxieme position des votants qui soutenaient $a$.
2. Verifiez le comportement sur votre profil : que retourne `Stv` ?
3. Une variante de Hare au sens strict serait-elle preferable pedagogiquement ?


In [7]:
// Exercice 1 -- a completer
using System;
using System.Collections.Generic;

public static class Sc09Exercise1
{
    public static List<string> Resoudre()
    {
        // TODO etudiant : construire le profil demande et comparer Stv a la variante
        // de transfert fractionnaire strict de Hare.
        // Profil type : cinq bulletins (a,b,c), puis un bulletin (b,a,c).
        // Indice : appelez Sc09Stv.Stv(RE, 2) et affichez le comite obtenu.
        Console.WriteLine("Exercice 1 (STV par transfert fractionnaire vs elimination) : a completer");
        return new List<string>();  // valeur neutre jusqu'a l'implementation etudiante
    }
}

var elusExercice1 = Sc09Exercise1.Resoudre();


Exercice 1 (STV par transfert fractionnaire vs elimination) : a completer


### Exercice 2 -- Complexite Monroe : pourquoi le backtracking explose

La regle de Monroe est NP-dure : la verification d'un comite exige de
chercher une affectation equilibree (backtracking sur les quotas
$[\lfloor n/k \rfloor, \lceil n/k \rceil]$), ce qui est en $O(k^n)$ --
pas $O(n)$ comme CC qui se contente d'un top-choice dans $W$ (definition
Chamberlin-Courant, equilibre non impose). La recherche complete est
donc $O(C(m, k) \cdot k^n)$ en worst case, impraticable des que $m$ depasse
15-20.

1. Mesurez le temps de `Monroe(R, k)` sur 5 profils aleatoires de tailles croissantes (`n in [4, 6]`, `m in [4, 6]`, `k in [2, 3]`). Utilisez `System.Diagnostics.Stopwatch`.
2. A partir de quel couple `(m, k)` le temps depasse-t-il 1 seconde ?
3. Pourquoi `Monroe` ne peut-il pas etre utilise tel quel pour des elections reelles (mairie de Strasbourg, 90 conseillers) ?

**Note pedagogique** : `Monroe(R, k)` inferera `m` depuis `R[0]` si non
fourni (cf. `CcBrute(R, k, m)` qui adopte la meme convention, `m` nullable).
Pour les profils jouets, l'inference est fiable.


In [8]:
// Exercice 2 -- a completer
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Linq;

public static class Sc09Exercise2
{
    public static List<string> Resoudre()
    {
        // TODO etudiant : mesurer le temps de Sc09Monroe.Monroe sur 5 profils aleatoires
        // de tailles croissantes (n in [4,6], m in [4,6], k in [2,3]).
        // Indice : utilisez System.Diagnostics.Stopwatch (Start / Elapsed.TotalSeconds)
        // autour de l'appel Monroe, puis repondez aux questions 2 et 3 de l'enonce.
        var resultats = new List<string>();   // valeur neutre : un chrono par profil
        Console.WriteLine("Exercice 2 (complexite Monroe) : a completer");
        return resultats;
    }
}

var chronoExercice2 = Sc09Exercise2.Resoudre();


Exercice 2 (complexite Monroe) : a completer


### Exercice 3 -- CC sous score approval vs Borda

L'implementation Chamberlin-Courant utilise le score Borda. Une variante classique utilise l'**approval** : chaque votant gagne 1 point par candidat elu qu'il approuve (dans un profil approval, pas ordonne). La complexite reste NP-dure (procaccia et al. 2008) mais l'enumeration directe peut etre plus rapide.

1. Convertissez le fil rouge en profil approval : un votant approuve ses 2 premiers candidats.
2. Ecrivez `CcApproval(RApp, k)` qui maximise le nombre d'approbations.
3. Comparez le comite CC-approval au comite CC-Borda sur le fil rouge. Sont-ils identiques ? Si non, quel candidat gagne ou perd ? Pourquoi ?


In [9]:
// Exercice 3 -- a completer
using System;
using System.Collections.Generic;
using System.Linq;

public static class Sc09Exercise3
{
    /// <summary>
    /// Convertit un profil ordonne en profil d'approbation : chaque votant approuve
    /// ses `nbApproves` premiers candidats.
    /// </summary>
    public static List<HashSet<string>> VersApproval(List<string[]> R, int nbApproves)
    {
        // TODO etudiant : renvoyer un profil d'approbation a partir du profil ordonne
        // (chaque votant approuve ses `nbApproves` premiers candidats).
        // Stub comme le jumeau Python : a l'etudiant d'ecrire la conversion.
        return null;   // valeur neutre jusqu'a l'implementation etudiante
    }

    /// <summary>
    /// Chamberlin-Courant sous score approval : maximise le nombre d'approbations.
    /// Renvoie (W, nombre d'approbations) ; W est null si aucune solution.
    /// </summary>
    public static (List<string> W, int approbations) CcApproval(List<HashSet<string>> RApp, int k)
    {
        // TODO etudiant : enumerer les comites de taille k et maximiser le total
        // des approbations (chaque votant compte 1 point par candidat elu qu'il approuve).
        Console.WriteLine("Exercice 3 (CC sous score approval) : a completer");
        return (null, 0);   // valeur neutre jusqu'a l'implementation etudiante
    }
}

var RAppEx3 = Sc09Exercise3.VersApproval(Sc09.R, 2);
if (RAppEx3 == null)
{
    Console.WriteLine("Exercice 3 (VersApproval) : a completer — profil d'approbation non calcule (valeur neutre).");
}
else
{
    var (WccApproval, scoreApproval) = Sc09Exercise3.CcApproval(RAppEx3, Sc09.K);
    Console.WriteLine($"CC approval : W = {(WccApproval == null ? "(a completer)" : Sc09.Fmt(WccApproval))}, approbations = {scoreApproval}");
}

Exercice 3 (VersApproval) : a completer — profil d'approbation non calcule (valeur neutre).


## Conclusion

- **STV (Hare, 1857)** est une regle multi-winners **polynomiale** au quota de Droop : un candidat est elu des qu'il atteint le quota (surplus transfere fractionnairement, methode de Gregory), et l'elimination du moins bien classe ne se declenche qu'en l'absence de candidat au quota (transfert integral, decompte constant). Le comite est complete par les **survivants** : il a toujours exactement $k$ membres, et son ordre ne depend pas de l'ordre d'elimination. Utilisee en Australie, en Irlande et pour le senat de Cambridge (MA).
- **Monroe (1995)** est une regle proportionnelle stricte : chaque elu represente un nombre equilibre de votants, $|\sigma^{-1}(c)| \in \{\lfloor n/k \rfloor, \lceil n/k \rceil\}$, et le score est calcule sur l'affectation assignee. **NP-dure** : l'enumeration conjointe comite/affectation est impraticable au-dela de $m=15$. Les algorithmes appropries sont la PLNE (MIP) et les approximations a garantie $1 - 1/e$.
- **Chamberlin-Courant (1983)** maximise la satisfaction agregee (ici Borda) du comite, sans contrainte d'equilibrage des representes. **NP-dure** sous Borda ou approval. Le solveur SOTA est la PLNE, implementable via `Microsoft.Z3` `Optimize` ou un solveur commercial type Gurobi.
- **Z3 confrontation** : sur les profils jouets (n <= 6, m <= 5), Z3 retrouve toujours l'optimum exact de l'enumeration directe -- garantie empirique que la formulation encode bien le probleme. Pour les profils plus grands, Z3 reste exact (jusqu'a la limite memoire) tandis que l'enumeration directe devient impraticable.

**Ce que l'experimentation mesure.** Sur 20 profils aleatoires uniformes ($n \in [4,6]$, $m \in [3,5]$, $k \in [2,3]$) generes par le carnet, les taux d'accord affiches en section 6 mesurent l'ecart entre les trois regles : la contrainte d'equilibrage de Monroe ne coute presque rien en satisfaction Borda sur ces profils, et le transfert de voix du STV (surplus puis eliminations successives) aboutit le plus souvent au meme comite que l'optimisation d'une affectation, l'ecart se concentrant sur les profils fractionnes. Sur les profils structures (fil rouge, blocs symetriques), les trois regles elisent le meme comite.

**Une precaution de mesure.** Ces taux d'accord ne valent que si les comites compares ont **le meme cardinal**. Une regle qui rendrait un comite de taille inferieure a $k$ ne pourrait jamais etre identique a une regle qui rend exactement $k$ membres : l'ecart mesure serait alors un artefact de taille et non une divergence de fond. C'est pourquoi la regle STV de la section 2 garantit un comite de cardinal exactement $k$ et **canonique** (trie) -- l'ordre d'elimination n'est pas une donnee du scrutin, et le resultat ne doit pas en dependre. Une regle dont le resultat n'a pas de forme canonique n'est pas comparable aux autres. La section 6 verifie cette propriete sur 200 profils aleatoires.

Le carnet prolonge SC-07 (core electoral) et SC-08 (Kemeny / Ranked Pairs / Dodgson) : meme structure pedagogique, meme ouvrage (*Handbook of Computational Social Choice*, **ch. 9** Lang et Xia, p. 207-210). Le carnet suivant (SC-10) couvrira la manipulation strategique sous regles multi-vainqueurs (Conitzer et al. 2010, Gibbard-Satterthwaite etendu aux comites).

**References.** Lang, J. & Xia, L. (2016). *Voting in Combinatorial Domains*. In *Handbook of Computational Social Choice* (Brandt, Conitzer, Endriss, Lang & Procaccia ed.), ch. 9, pp. 197-252. Cambridge University Press. Ancrage du carnet : p. 207-210 pour Chamberlin-Courant, Monroe, PLNE et NP-completude. ISBN 978-1-107-06043-2 ; Elkind, E., Faliszewski, P., Skowron, P. & Slinko, A. (2017). *Properties of Multiwinner Voting Rules*. Social Choice and Welfare 48(3):599-632 (article, pas un chapitre du Handbook -- le Handbook le cite sous "Elkind et al. 2014a") ; Monroe, B.L. (1995). *Fully Proportional Representation*. American Political Science Review 89(4):925-940 ; Chamberlin, J. & Courant, P. (1983). *Representative Deliberations* ; Brams, S. & Potthoff, R. (1998). *Utilitarian Fairness in Committee Selection* (resolution PLNE de Chamberlin-Courant) ; Procaccia, A., Rosenschein, J. & Zohar, A. (2008). *On the complexity of achieving proportional representation* (Social Choice and Welfare, NP-completude) ; Lu, T. & Boutilier, C. (2011a). *Budgeted Social Choice* (NP-completude, cadre budgeted). Notebooks voisins : 01-Arrow, ChoixSocial-03-Voting-Methods, ChoixSocial-04-SAT-Z3, ChoixSocial-07-Committees-Core, ChoixSocial-08-Kemeny-RankedPairs-Dodgson.

> Carnet issu de l'EPIC #26534.
